In [5]:
import os
import io
import pandas as pd
import warnings

from os import walk
from ase.io import read, write
# from molgeom import Molecule
from rdkit import Chem
from tqdm import tqdm
from openbabel import openbabel
from molSimplify.Classes import mol3D
from rdkit import RDLogger  

warnings.filterwarnings("ignore")
RDLogger.DisableLog('rdApp.*')   
openbabel.obErrorLog.SetOutputLevel(0)

In [73]:

coordination_geometries = {
    2: ["linear", "bent"],
    3: ["trigonal planar", "trigonal pyramidal", "T shape"],
    4: ["tetrahedral", "square planar", "seesaw", "square tetrahedral"],
    5: ["trigonal bipyramidal", "square pyramidal", "pentagonal planar", "trigonal prismatic"],
    6: ["octahedral", "trigonal prismatic", "pentagonal pyramidal", "hexagonal planar", "trigonal antiprismatic"],
    7: ["pentagonal bipyramidal", "capped octahedral", "monocapped trigonal prismatic", "capped trigonal prismatic"],
    8: ["square antiprismatic", "dodecahedral", "bicapped trigonal prismatic", "hexagonal bipyramidal", "cube"],
    9: ["tricapped trigonal prismatic", "capped square antiprismatic", "tricapped octahedral"],
    10: ["bicapped square antiprismatic", "pentagonal prismatic", "decagonal bipyramidal"],
    11: ["octahedral face-capped", "capped pentagonal bipyramidal", "monocapped bicapped square antiprismatic"],
    12: ["icosahedral", "cuboctahedral", "hexagonal prismatic", "square cuboctahedral"]
}

def find_coord_no(geo_name):    
    for t, k in coordination_geometries.items():      
        # print(k)  
        for cur_geo in k:
            if geo_name.lower() == cur_geo.lower():
                return t

def get_all_unique_geometries():
    tl = []
    for t, k in coordination_geometries.items():      
        # print(k)  
        for cur_geo in k:
            tl.append(cur_geo)  
    
    return [x for x in set(tl)]


In [74]:
def get_geometry_from_mols(molsimp_mol):    
    ret_geo = ("unknown", 999999)
    
    if molsimp_mol is not None:
        cur_geo = molsimp_mol.get_geometry_type()
        # print("geo", cur_geo)
        if "geometry" in cur_geo:
            ret_geo = (cur_geo["geometry"], cur_geo["angle_devi"],  find_coord_no(cur_geo["geometry"]))
    
    return ret_geo

def get_geometry_from_rdkit(filename):
    ret_geo = ("unknown", 0)
    
    try:
        rd_mol = Chem.MolFromMolFile(filename)      
        set_ats = set([x.GetSymbol() for x in rd_mol.GetAtoms()])
        # print(set_ats)
        if "R" in set_ats or "*" in set_ats:
            return ret_geo

        # problems = Chem.DetectChemistryProblems(rd_mol)
        # print(problems)
        if rd_mol is None:
            
            return ret_geo
        Chem.AddHs(rd_mol)
        Chem.SanitizeMol(rd_mol)      
        xyz_string = Chem.MolToXYZBlock(rd_mol)               
        complex_mol = mol3D()
        complex_mol.readfromstring(xyz_string)    
        ret_geo = get_geometry_from_mols(complex_mol)
        
    except Exception as ex1:
        # print("ERROR rdkit", str(ex1))  
        pass 
    
    return ret_geo      

def get_geometry_from_openbabel(filename):    
    ret_geo = ("unknown", 0)
    
    try:
        tmp_mol = openbabel.OBMol()
        obConversion = openbabel.OBConversion()
        obConversion.SetInFormat("mol")
        obConversion.SetOutFormat("xyz")
        
        if obConversion.ReadFile(tmp_mol, filename):
            xyz_string = obConversion.WriteString(tmp_mol)
            
            complex_mol = mol3D()
            try:
                complex_mol.readfromstring(xyz_string)                                                
                ret_geo = get_geometry_from_mols(complex_mol)    
            except:
                pass
                                    
    except Exception as ex2:
        print("ERROR openbabel", str(ex2))      
    
    return ret_geo 
    
def get_geometry_from_molsimplify(filename):
    ret_geo = ("unknown", 0)
    
    try:
        complex_mol = mol3D()
        complex_mol.readfrommol(filename)

        ret_geo = get_geometry_from_mols(complex_mol)
        
    except Exception as ex3:
        # print("ERROR molsimplify", str(ex3))    
        pass  
        
    return ret_geo
        
def get_geometry_from_ase(filename):
    ret_geo = ("unknown", 0)
    
    try:
        
        ase_mol_atoms = read(filename, format='mol')
        
        ase_file = io.StringIO()
        write(ase_file, ase_mol_atoms, format='xyz')
        xyz_string = ase_file.getvalue()
        complex_mol = mol3D()
        complex_mol.readfromstring(xyz_string)    
        ret_geo = get_geometry_from_mols(complex_mol)
            
        ase_file.close()
        
    except Exception as ex5:
        # print("ERROR ase", str(ex5))    
        pass
    
    return ret_geo   

def get_geometry_from_file(input_filename, load_type="all"):
    
    id_geometry = "unknown"
    try:
        
        tl = []
        if load_type == "all" or load_type == "rdkit":
            tl.append(get_geometry_from_rdkit(input_filename))
        
        if load_type == "all" or load_type == "openbabel":
            tl.append(get_geometry_from_openbabel(input_filename))
            
        if load_type == "all" or load_type == "molsimplify":
            tl.append(get_geometry_from_molsimplify(input_filename))
        
        if load_type == "all" or load_type == "ase":
            tl.append(get_geometry_from_ase(input_filename))
                
        tl_geo = [x for x in tl if x[0] != "unknown"]
        
        # print(tl_geo)
        
        if len(tl_geo) > 0:            
            # print(tl_geo)
            tl_geo.sort(key=lambda x: x[2], reverse=True)
            
            # print(tl_geo)
            id_geometry = tl_geo[0][0]
                
    except Exception as ex_tt:
        print(str(ex_tt))
    
    return id_geometry



In [ ]:
# filename = "../data/AlexClarkStructures_v1/training.ds/mol_training.ds_543.mol"
# print(get_geometry_from_file(filename))
# filename = '../data/AlexClarkStructures_v2/training.ds/mol_training.ds_25.mol'
# print(get_geometry_from_file(filename))
# filename = '../data/AlexClarkStructures_v2/training.ds/mol_training.ds_482.mol'
# print(get_geometry_from_file(filename))
# filename = '../data/AlexClarkStructures_v1/training.ds/mol_training.ds_83.mol'
# print(get_geometry_from_file(filename))
# filename = '../data/AlexClarkStructures_v1/training.ds/mol_training.ds_47.mol'
# print(get_geometry_from_file(filename))
# filename = '../data/AlexClarkStructures_v1/curated-0001.ds/mol_curated-0001.ds_0.mol'
# print(get_geometry_from_file(filename))
# filename = '../data/AlexClarkStructures_v1/formatelements.ds/mol_formatelements.ds_4.mol'
# print(get_geometry_from_file(filename, "openbabel"))

# filename = '../data/AlexClarkStructures_v1/curated-0001.ds/mol_curated-0001.ds_115.mol'
# print(get_geometry_from_file(filename, "openbabel"))



filename = '../data/AlexClarkStructures_v2/curated-selection.ds/mol_curated-selection.ds_314.mol'
print(get_geometry_from_file(filename, "openbabel"))


In [ ]:

# folder path
dir_path = '../data/AlexClarkStructures_v2/'

tl_ld = ["all", "rdkit", "openbabel", "ase", "molsimplify"]
# tl_ld = ["openbabel"]

dict_res = {}

for cur_ld in tl_ld:
    cur_geo_dict = {x : 0 for x in get_all_unique_geometries()}
    # list to store files name
    for path, subdirs, files in os.walk(dir_path):
        for name in files:
            cur_filename = os.path.join(path, name)
            # print(cur_filename)
            cur_geo = get_geometry_from_file(cur_filename, load_type=cur_ld)
            if cur_geo not in cur_geo_dict:
                # cur_geo_dict[cur_geo] = []
                cur_geo_dict[cur_geo] = 0
                
            cur_geo_dict[cur_geo] += 1 #.append(cur_filename)  
               
    
    dict_res[cur_ld] = cur_geo_dict
    
df_res = pd.DataFrame(dict_res)

In [79]:
df_res.fillna(0, inplace=True)
df_res = df_res.loc[~(df_res==0).all(axis=1)] 
df_res.sort_index(inplace=True)
df_res.astype(int)

,all,rdkit,openbabel,ase,molsimplify
T shape,132,81,132,0,0
linear,537,411,532,0,0
octahedral,55,44,55,0,0
seesaw,59,30,61,0,0
square planar,407,250,407,0,0
square pyramidal,1,1,1,0,0
tetrahedral,3,2,3,0,0
trigonal bipyramidal,44,28,42,0,0
trigonal planar,268,165,268,0,0
trigonal prismatic,2,0,2,0,0
